# NexoServicios · Clase 2: procesamiento de datos con PL/SQL

**La situación.** NexoServicios recibe cada mañana un **archivo de órdenes de servicio** de un tercero y tiene que cargarlo en su base. El archivo trae errores: montos imposibles, canales mal escritos, clientes que no existen, órdenes repetidas. Hoy construimos el **procedimiento** que lo carga y lo hacemos **observable, transaccional y verificable**.

> **El criterio de éxito, desde el inicio:** al terminar la corrida, una consulta de conciliación tiene que demostrar que **filas leídas = filas cargadas + filas rechazadas**, y que nada quedó cargado a medias.

Y una **segunda situación** (Pasos 10 a 12): tres versiones de un procedimiento que actualiza la prioridad de los clientes, para **decidir cuál usar y por qué**.

**Cómo trabajamos.** Toda la clase es **práctica guiada**: de a un paso, con el código **comentado línea por línea**. Cada paso tiene tres momentos: *qué vamos a hacer*, la celda (se corre con `Shift` + `Enter`) y *qué vimos*.

| Tipo de celda | Qué hacés |
| --- | --- |
| **Ejecutamos juntos** | El código ya está escrito y comentado: lo leemos, predecimos y ejecutamos |
| **👉 Tu turno** | Modificás el código para resolver algo parecido (al final, el ejercicio por equipos) |

### ¿Qué se ejecuta de verdad? (nivel de realización)

El programa pide declarar el nivel de realización de cada componente. Acá es así:

| Componente | Nivel | Cómo |
| --- | --- | --- |
| **Datos** | Ejecutado con datos sintéticos | Se generan desde código, sin archivos que subir |
| **Procedimiento y pruebas** | **Simulado de forma reproducible** | Un equivalente en Python sobre SQLite, con la **misma estructura** que el PL/SQL (constantes, bloque anidado por fila, bitácora, COMMIT al final) |
| **PL/SQL de Oracle** | **Especificado, no ejecutado en este entorno** | Se muestra el código PL/SQL equivalente, escrito para implementarse; no hay una base Oracle dentro de este notebook |

> Lo que se aprende (el contrato, el manejo de errores, la idempotencia y la optimización medida) es el mismo en los dos mundos. Antes de usar el PL/SQL en una base real, probalo en tu entorno.

## Paso 0 · Preparamos las herramientas

Usamos solo tres librerías: **NumPy** (sorteos), **pandas** (tablas) y **matplotlib** (gráficos). La base de datos es **SQLite**, que ya viene con Python. La primera celda instala lo que falte.

In [1]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas", "matplotlib")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

✅ Librerías listas


In [2]:
import sqlite3                  # la base de datos que viene con Python
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.width = 140
pd.options.display.max_columns = 20
print("python ✔ · pandas:", pd.__version__, "· numpy:", np.__version__, "· sqlite:", sqlite3.sqlite_version)

python ✔ · pandas: 3.0.6 · numpy: 2.5.3 · sqlite: 3.45.1


## Paso 1 · Los datos de NexoServicios

Tres cosas viven en la base: los **clientes**, las **órdenes que ya están cargadas** y las tres tablas de control que va a escribir el procedimiento (**rechazos** y **bitácora**; las armamos en el Paso 3). Y llega un **archivo**: las órdenes nuevas, tal como las mandó el tercero.

### 1.1 · La semilla y las reglas del negocio
Casi todo lo que sigue es un sorteo. La **semilla** hace que todos obtengamos los mismos datos. Y declaramos **una sola vez** las reglas de negocio: son los parámetros del procedimiento.

In [3]:
rng = np.random.default_rng(2026)           # la semilla: el «número de sorteo»

# --- Reglas de negocio (se declaran una sola vez, arriba) ---
CANALES_VALIDOS = ("WEB", "APP", "TELEFONO", "PRESENCIAL")
MONTO_MIN, MONTO_MAX = 1, 50_000
UMBRAL_RECHAZO = 0.20                       # si se rechaza más del 20 % del archivo, la corrida se detiene
print("Canales válidos:", CANALES_VALIDOS, "· monto entre", MONTO_MIN, "y", MONTO_MAX, "· umbral de rechazo:", UMBRAL_RECHAZO)

Canales válidos: ('WEB', 'APP', 'TELEFONO', 'PRESENCIAL') · monto entre 1 y 50000 · umbral de rechazo: 0.2


### 1.2 · Clientes y órdenes ya cargadas
500 clientes y 1.000 órdenes de cargas anteriores (`id_carga = 0`).

In [4]:
clientes = pd.DataFrame({"id_cliente": [f"C{i:04d}" for i in range(1, 501)],
                         "segmento": rng.choice(["Residencial", "Empresa", "Gobierno"], size=500, p=[0.6, 0.3, 0.1])})

ordenes_previas = pd.DataFrame({
    "id_orden": [f"OS{i:07d}" for i in range(1, 1001)],
    "id_cliente": rng.choice(clientes["id_cliente"], size=1000),
    "canal": rng.choice(CANALES_VALIDOS, size=1000),
    "monto": np.round(rng.uniform(100, 20_000, size=1000), 2),
    "fecha_alta": "2025-09-30", "id_carga": 0})
clientes.head(3)

,id_cliente,segmento
0,C0001,Residencial
1,C0002,Empresa
2,C0003,Residencial


### 1.3 · El archivo que llega
Una función genera el archivo con **errores plantados** (y guarda cuáles son, para poder comprobar después que el procedimiento los encuentra). Con `escala` controlamos cuántos errores trae.

In [5]:
def generar_archivo(n, escala=1.0, primer_id=1001, semilla=7):
    """Archivo de órdenes de un tercero, con errores de negocio plantados. Devuelve (archivo, defectos_plantados)."""
    g = np.random.default_rng(semilla)
    a = pd.DataFrame({
        "nro_fila": np.arange(1, n + 1),
        "id_orden": [f"OS{primer_id + i:07d}" for i in range(n)],
        "id_cliente": g.choice(clientes["id_cliente"], size=n),
        "canal": g.choice(CANALES_VALIDOS, size=n),
        "monto": np.round(g.lognormal(7.5, 0.8, size=n).clip(20, 20_000), 2),
        "fecha_alta": pd.to_datetime("2025-10-01") + pd.to_timedelta(g.integers(0, 31, size=n), unit="D")})
    a["fecha_alta"] = a["fecha_alta"].dt.strftime("%Y-%m-%d")
    plan = {"formato de canal (se corrige)": 0.04, "canal inexistente": 0.01, "monto fuera de rango": 0.015,
            "cliente inexistente": 0.01, "fecha inválida": 0.005, "orden repetida en el archivo": 0.01, "orden ya cargada": 0.005}
    libres = list(g.permutation(n))
    defectos = {}
    for nombre, p in plan.items():
        k = max(1, int(round(n * p * escala)))
        filas, libres = libres[:k], libres[k:]
        defectos[nombre] = len(filas)
        for i in filas:
            if nombre.startswith("formato"):
                a.loc[i, "canal"] = g.choice([" web", "App ", "telefono", "Presencial", " TELEFONO "])
            elif nombre == "canal inexistente":
                a.loc[i, "canal"] = g.choice(["FAX", "CORREO"])
            elif nombre == "monto fuera de rango":
                a.loc[i, "monto"] = g.choice([-50.0, 0.0, 120_000.0, 999_999.0])
            elif nombre == "cliente inexistente":
                a.loc[i, "id_cliente"] = "C9999"
            elif nombre == "fecha inválida":
                a.loc[i, "fecha_alta"] = g.choice(["sin fecha", "2025-13-40"])
            elif nombre == "orden repetida en el archivo":
                a.loc[i, "id_orden"] = a.loc[max(0, i - 3), "id_orden"] if i > 3 else a.loc[i + 3, "id_orden"]
            elif nombre == "orden ya cargada":
                a.loc[i, "id_orden"] = "OS0000100"
    return a, pd.Series(defectos, name="plantados")

archivo, plantados = generar_archivo(5_000)                 # el archivo «habitual»: pocos errores
archivo_malo, _ = generar_archivo(5_000, escala=6.0, primer_id=10_001, semilla=9)   # otro archivo, con muchísimos errores
print(f"Archivo habitual: {len(archivo):,} filas · archivo malo: {len(archivo_malo):,} filas")
archivo.head()

Archivo habitual: 5,000 filas · archivo malo: 5,000 filas


,nro_fila,id_orden,id_cliente,canal,monto,fecha_alta
0,1,OS0001001,C0473,TELEFONO,2543.26,2025-10-16
1,2,OS0001002,C0313,TELEFONO,3527.79,2025-10-16
2,3,OS0001003,C0343,TELEFONO,3328.15,2025-10-13
3,4,OS0001004,C0449,TELEFONO,1420.37,2025-10-15
4,5,OS0001005,C0290,WEB,2554.29,2025-10-16


**Los errores plantados** en el archivo habitual (el procedimiento tiene que encontrarlos):

In [6]:
plantados.to_frame()

,plantados
formato de canal (se corrige),200
canal inexistente,50
monto fuera de rango,75
cliente inexistente,50
fecha inválida,25
orden repetida en el archivo,50
orden ya cargada,25


**Qué vimos:** el archivo trae los errores típicos de un tercero. Algunos se **corrigen** (un canal escrito con espacios o minúsculas, que se normaliza) y otros se **rechazan** (un monto imposible, un cliente que no existe).

## Paso 2 · Primero, el criterio de éxito

Antes de escribir una línea del procedimiento, definimos **qué consultas van a demostrar que la corrida funcionó**. Si no podemos decirlo, todavía no sabemos qué estamos construyendo.

| # | Consulta de conciliación | Qué demuestra |
| --- | --- | --- |
| 1 | `leídas = cargadas + rechazadas` | No se perdió ninguna fila |
| 2 | No hay `id_orden` repetidos en `ordenes` | No se duplicó nada |
| 3 | Todos los canales cargados son válidos | La corrección funcionó |
| 4 | Todos los montos están en rango | Las reglas se aplicaron |
| 5 | Todo rechazo tiene un motivo | Se puede explicar cada fila que quedó afuera |

Las dejamos escritas como una función: **la vamos a usar al final de cada corrida**.

In [7]:
def verificar_corrida(con, id_carga, leidas):
    """Las consultas de conciliación del Paso 2. Devuelve una tabla con ✅ o ❌ por control."""
    q = lambda sql, *p: con.execute(sql, p).fetchone()[0]
    cargadas = q("SELECT COUNT(*) FROM ordenes WHERE id_carga = ?", id_carga)
    rechazadas = q("SELECT COUNT(*) FROM rechazos WHERE id_carga = ?", id_carga)
    canales = ",".join(f"'{x}'" for x in CANALES_VALIDOS)
    controles = [
        ("1 · leídas = cargadas + rechazadas", leidas == cargadas + rechazadas, f"{leidas:,} = {cargadas:,} + {rechazadas:,}"),
        ("2 · sin id_orden repetidos", q("SELECT COUNT(*) - COUNT(DISTINCT id_orden) FROM ordenes") == 0, "0 repetidos"),
        ("3 · canales válidos", q(f"SELECT COUNT(*) FROM ordenes WHERE canal NOT IN ({canales})") == 0, "0 inválidos"),
        ("4 · montos en rango", q("SELECT COUNT(*) FROM ordenes WHERE monto NOT BETWEEN ? AND ?", MONTO_MIN, MONTO_MAX) == 0, "0 fuera de rango"),
        ("5 · todo rechazo tiene motivo", q("SELECT COUNT(*) FROM rechazos WHERE id_carga = ? AND (motivo IS NULL OR motivo = '')", id_carga) == 0, "0 sin motivo")]
    return pd.DataFrame([(("✅" if ok else "❌"), n, d) for n, ok, d in controles], columns=["", "control", "detalle"])
print("Consultas de conciliación definidas ✔")

Consultas de conciliación definidas ✔


## Paso 3 · Las tablas que escribe el procedimiento

Además de `ordenes` (el destino), el procedimiento escribe dos tablas de **control**: `rechazos` (qué filas quedaron afuera y por qué) y `bitacora_cargas` (qué pasó en cada corrida). Las restricciones de la tabla (`PRIMARY KEY`, `CHECK`) son una **segunda línea de defensa**: si la lógica falla, la base igual protege los datos.

In [8]:
DDL = """
CREATE TABLE clientes (id_cliente TEXT PRIMARY KEY, segmento TEXT);
CREATE TABLE ordenes (
    id_orden   TEXT PRIMARY KEY,                                              -- una orden, una sola vez
    id_cliente TEXT NOT NULL REFERENCES clientes(id_cliente),
    canal      TEXT NOT NULL CHECK (canal IN ('WEB','APP','TELEFONO','PRESENCIAL')),
    monto      REAL NOT NULL CHECK (monto BETWEEN 1 AND 50000),
    fecha_alta TEXT NOT NULL,
    id_carga   INTEGER NOT NULL);
CREATE TABLE rechazos (id_carga INTEGER, nro_fila INTEGER, id_orden TEXT, motivo TEXT NOT NULL);
CREATE TABLE bitacora_cargas (
    id_carga INTEGER PRIMARY KEY, archivo TEXT, inicio TEXT, fin TEXT,
    leidas INTEGER, cargadas INTEGER, rechazadas INTEGER, corregidas INTEGER, estado TEXT, detalle TEXT);
"""

def nueva_base():
    """Una base de datos nueva, con las tablas y los datos de partida. (isolation_level=None: las transacciones las controlamos nosotros.)"""
    con = sqlite3.connect(":memory:", isolation_level=None)
    con.execute("PRAGMA foreign_keys = ON")                 # la base también protege la integridad (segunda línea de defensa)
    con.executescript(DDL)
    clientes.to_sql("clientes", con, if_exists="append", index=False)
    ordenes_previas.to_sql("ordenes", con, if_exists="append", index=False)
    return con

con = nueva_base()
print("Tablas:", [t[0] for t in con.execute("SELECT name FROM sqlite_master WHERE type = 'table'")])
print("Órdenes ya cargadas:", con.execute("SELECT COUNT(*) FROM ordenes").fetchone()[0])

Tablas: ['clientes', 'ordenes', 'rechazos', 'bitacora_cargas']
Órdenes ya cargadas: 1000


**Qué vimos:** quedó una base con las órdenes de antes y las tablas de control vacías. Desde acá, cada corrida es una función que recibe el archivo y devuelve un resumen.

## Paso 4 · Las fallas de negocio y su respuesta

Antes de codificar, **definimos explícitamente cada posible falla de negocio** y qué hace el procedimiento con ella: **rechazo** (la fila no se carga y queda registrada), **corrección** (se arregla y se carga, anotando que se corrigió) o **detención** (se frena toda la corrida).

| Falla | Respuesta | Por qué |
| --- | --- | --- |
| Canal escrito con espacios o minúsculas (` web`, `App `) | **Corrección** | El dato es reconocible: se normaliza |
| Canal que no existe (`FAX`) | **Rechazo** | No hay forma segura de adivinar |
| Monto fuera de rango | **Rechazo** | Un monto imposible no se corrige solo |
| Cliente inexistente | **Rechazo** | Rompería la integridad de la base |
| Fecha inválida | **Rechazo** | No se puede ubicar la orden en el tiempo |
| Orden repetida (en el archivo o ya cargada) | **Rechazo** | Duplicaría la facturación |
| Más del 20 % del archivo rechazado | **Detención** | Algo anda mal en el origen: mejor frenar y revisar |

Las reglas se evalúan **siempre en el mismo orden**. Así cada fila rechazada tiene un solo motivo.

In [ ]:
class ErrorNegocio(Exception):
    """Falla de negocio de UNA fila: se rechaza esa fila y la corrida sigue."""
    def __init__(self, motivo):
        super().__init__(motivo)
        self.motivo = motivo

class UmbralSuperado(Exception):
    """Falla de la corrida entera: se detiene todo."""

def validar_fila(fila, existe_cliente, existe_orden, vistos):
    """Aplica las reglas de negocio a UNA fila, siempre en el mismo orden. Devuelve (fila normalizada, ¿se corrigió el canal?)."""
    try:                                                   # 1 · la fecha tiene que ser una fecha de verdad
        datetime.strptime(str(fila["fecha_alta"]), "%Y-%m-%d")
    except ValueError:
        raise ErrorNegocio("FECHA_INVALIDA")
    if not (MONTO_MIN <= fila["monto"] <= MONTO_MAX):      # 2 · monto en rango
        raise ErrorNegocio("MONTO_FUERA_DE_RANGO")
    canal = str(fila["canal"]).strip().upper()             # 3 · canal: se normaliza, y tiene que ser uno de los válidos
    if canal not in CANALES_VALIDOS:
        raise ErrorNegocio("CANAL_INCORRECTO")
    if not existe_cliente(fila["id_cliente"]):             # 4 · el cliente tiene que existir
        raise ErrorNegocio("CLIENTE_INEXISTENTE")
    if fila["id_orden"] in vistos:                         # 5 · la orden no puede repetirse dentro del archivo...
        raise ErrorNegocio("ORDEN_REPETIDA_EN_ARCHIVO")
    if existe_orden(fila["id_orden"]):                     # 6 · ...ni estar ya cargada
        raise ErrorNegocio("ORDEN_YA_CARGADA")
    return {**fila, "canal": canal}, canal != fila["canal"]
print("Reglas de negocio definidas ✔ (6 reglas, siempre en el mismo orden)")

## Paso 5 · El procedimiento, de punta a punta

Seguimos el **procedimiento de uso de PL/SQL**, en orden:

1. **Criterio de éxito** (ya lo escribimos en el Paso 2).
2. **Parámetros variables** (identificador de carga, nombre de archivo, umbral) declarados **arriba**, como constantes o variables.
3. **`%TYPE` y `%ROWTYPE`** para anclar las variables de datos al esquema real.
4. **El camino esperado primero**, antes de agregar las ramas de error.
5. **Cada falla de negocio**, con su respuesta (Paso 4).
6. **Un bloque anidado con `EXCEPTION` alrededor de la carga de cada fila**.
7. **Bitácora y un solo `COMMIT` al final**, nunca dentro del bucle.
8. **Conciliación como validación final**.

Este es el procedimiento en **PL/SQL de Oracle** (especificado, no ejecutado acá). Fijate cómo cada comentario apunta a uno de los ocho pasos:

```sql
-- La bitácora tiene que sobrevivir al ROLLBACK de la carga: por eso es una transacción autónoma.
CREATE OR REPLACE PROCEDURE pr_bitacora (
    p_id_carga IN bitacora_cargas.id_carga%TYPE, p_estado IN VARCHAR2, p_detalle IN VARCHAR2,
    p_leidas IN PLS_INTEGER, p_cargadas IN PLS_INTEGER, p_rechazadas IN PLS_INTEGER
) IS
    PRAGMA AUTONOMOUS_TRANSACTION;
BEGIN
    UPDATE bitacora_cargas
       SET estado = p_estado, detalle = p_detalle, fin = SYSDATE,
           leidas = p_leidas, cargadas = p_cargadas, rechazadas = p_rechazadas
     WHERE id_carga = p_id_carga;
    COMMIT;                                        -- confirma solo la bitácora
END pr_bitacora;
/

CREATE OR REPLACE PROCEDURE pr_cargar_ordenes (
    p_id_carga       IN  bitacora_cargas.id_carga%TYPE,     -- (2) parámetros, anclados al esquema
    p_umbral_rechazo IN  NUMBER DEFAULT 0.20,
    p_leidas         OUT PLS_INTEGER,                       -- contrato: lo que devuelve
    p_cargadas       OUT PLS_INTEGER,
    p_rechazadas     OUT PLS_INTEGER
) IS
    c_monto_min  CONSTANT ordenes.monto%TYPE := 1;          -- (2) constantes arriba del bloque
    c_monto_max  CONSTANT ordenes.monto%TYPE := 50000;
    v_canal      ordenes.canal%TYPE;                        -- (3) %TYPE
    v_fecha      DATE;
    v_existe     PLS_INTEGER;
    v_motivo     rechazos.motivo%TYPE;
    e_negocio          EXCEPTION;                           -- (5) falla de UNA fila
    e_umbral_superado  EXCEPTION;                           --     falla de la corrida entera
BEGIN
    p_leidas := 0; p_cargadas := 0; p_rechazadas := 0;

    FOR r IN (SELECT * FROM stg_ordenes WHERE id_carga = p_id_carga ORDER BY nro_fila) LOOP   -- (3) r es %ROWTYPE
        p_leidas := p_leidas + 1;
        BEGIN                                                -- (6) bloque anidado: una falla afecta solo a esta fila
            BEGIN
                v_fecha := TO_DATE(r.fecha_alta, 'YYYY-MM-DD');
            EXCEPTION WHEN OTHERS THEN
                v_motivo := 'FECHA_INVALIDA'; RAISE e_negocio;
            END;
            IF r.monto NOT BETWEEN c_monto_min AND c_monto_max THEN
                v_motivo := 'MONTO_FUERA_DE_RANGO'; RAISE e_negocio;
            END IF;
            v_canal := UPPER(TRIM(r.canal));                 -- corrección: se normaliza
            IF v_canal NOT IN ('WEB', 'APP', 'TELEFONO', 'PRESENCIAL') THEN
                v_motivo := 'CANAL_INCORRECTO'; RAISE e_negocio;
            END IF;
            SELECT COUNT(*) INTO v_existe FROM clientes WHERE id_cliente = r.id_cliente;
            IF v_existe = 0 THEN
                v_motivo := 'CLIENTE_INEXISTENTE'; RAISE e_negocio;
            END IF;

            INSERT INTO ordenes (id_orden, id_cliente, canal, monto, fecha_alta, id_carga)     -- (4) el camino esperado
            VALUES (r.id_orden, r.id_cliente, v_canal, r.monto, r.fecha_alta, p_id_carga);
            p_cargadas := p_cargadas + 1;
        EXCEPTION
            WHEN e_negocio THEN                              -- se rechaza la fila y se sigue
                INSERT INTO rechazos (id_carga, nro_fila, id_orden, motivo) VALUES (p_id_carga, r.nro_fila, r.id_orden, v_motivo);
                p_rechazadas := p_rechazadas + 1;
            WHEN DUP_VAL_ON_INDEX THEN                       -- la clave primaria protege contra órdenes repetidas
                INSERT INTO rechazos (id_carga, nro_fila, id_orden, motivo) VALUES (p_id_carga, r.nro_fila, r.id_orden, 'ORDEN_DUPLICADA');
                p_rechazadas := p_rechazadas + 1;
        END;
    END LOOP;

    IF p_rechazadas > p_leidas * p_umbral_rechazo THEN       -- detención: se frena toda la corrida
        RAISE e_umbral_superado;
    END IF;

    COMMIT;                                                  -- (7) un solo COMMIT, al final
    pr_bitacora(p_id_carga, 'OK', NULL, p_leidas, p_cargadas, p_rechazadas);
EXCEPTION
    WHEN e_umbral_superado THEN
        ROLLBACK;
        pr_bitacora(p_id_carga, 'DETENIDA', 'Rechazos por encima del umbral', p_leidas, 0, p_rechazadas);
        RAISE_APPLICATION_ERROR(-20001, 'Carga detenida: rechazos por encima del umbral');
    WHEN OTHERS THEN
        ROLLBACK;
        pr_bitacora(p_id_carga, 'ERROR', SQLERRM, p_leidas, 0, p_rechazadas);
        RAISE;                                               -- no se silencia: se propaga
END pr_cargar_ordenes;
/

-- (8) Conciliación: filas leídas = cargadas + rechazadas
SELECT leidas, cargadas, rechazadas, leidas - (cargadas + rechazadas) AS diferencia
  FROM bitacora_cargas WHERE id_carga = :id_carga;
```

**Ahora, el equivalente ejecutable** (Python sobre SQLite), con **la misma estructura**. Leelo comparando con el PL/SQL de arriba: constantes, bloque anidado por fila, bitácora, `COMMIT` al final.

In [ ]:
def pr_cargar_ordenes(con, p_id_carga, p_archivo, filas, p_umbral=UMBRAL_RECHAZO, falla_en_fila=None):
    """Equivalente en Python del procedimiento PL/SQL. Devuelve el resumen (los parámetros OUT)."""
    hoy = datetime.now().isoformat(timespec="seconds")

    # --- Idempotencia: si esta carga ya terminó bien, no se hace nada ---
    previa = con.execute("SELECT estado, leidas, cargadas, rechazadas, corregidas FROM bitacora_cargas WHERE id_carga = ?", (p_id_carga,)).fetchone()
    if previa and previa[0] == "OK":
        return {"estado": "YA_EJECUTADA", "leidas": previa[1], "cargadas": previa[2], "rechazadas": previa[3], "corregidas": previa[4]}

    # --- La bitácora se escribe FUERA de la transacción de la carga (equivale a PRAGMA AUTONOMOUS_TRANSACTION) ---
    con.execute("INSERT OR REPLACE INTO bitacora_cargas (id_carga, archivo, inicio, estado) VALUES (?, ?, ?, 'EN_CURSO')", (p_id_carga, p_archivo, hoy))

    existe_cliente = lambda c: con.execute("SELECT 1 FROM clientes WHERE id_cliente = ?", (c,)).fetchone() is not None
    existe_orden = lambda o: con.execute("SELECT 1 FROM ordenes WHERE id_orden = ?", (o,)).fetchone() is not None
    leidas = cargadas = rechazadas = corregidas = 0
    vistos = set()
    try:
        con.execute("BEGIN")                                            # empieza la transacción de la carga
        for fila in filas:                                              # FOR r IN (SELECT ...) LOOP
            leidas += 1
            if falla_en_fila == leidas:
                raise RuntimeError("falla inesperada (simulada) en la fila %d" % leidas)
            try:                                                        # bloque anidado: una falla afecta solo a esta fila
                ok, corregida = validar_fila(fila, existe_cliente, existe_orden, vistos)
                con.execute("INSERT INTO ordenes (id_orden, id_cliente, canal, monto, fecha_alta, id_carga) VALUES (?,?,?,?,?,?)",
                            (ok["id_orden"], ok["id_cliente"], ok["canal"], ok["monto"], ok["fecha_alta"], p_id_carga))
                cargadas += 1
                corregidas += int(corregida)
            except ErrorNegocio as e:                                   # WHEN e_negocio THEN ...
                con.execute("INSERT INTO rechazos (id_carga, nro_fila, id_orden, motivo) VALUES (?,?,?,?)", (p_id_carga, fila["nro_fila"], fila["id_orden"], e.motivo))
                rechazadas += 1
            vistos.add(fila["id_orden"])
        if rechazadas > leidas * p_umbral:                              # detención: se frena toda la corrida
            raise UmbralSuperado()
        con.execute("COMMIT")                                           # (7) un solo COMMIT, al final
        estado, detalle = "OK", None
    except UmbralSuperado:
        con.execute("ROLLBACK"); cargadas = 0; estado, detalle = "DETENIDA", "Rechazos por encima del umbral"
    except Exception as e:
        con.execute("ROLLBACK"); cargadas = 0; estado, detalle = "ERROR", str(e)
        _cerrar_bitacora(con, p_id_carga, estado, detalle, leidas, cargadas, rechazadas, corregidas)
        raise                                                           # RAISE: no se silencia, se propaga
    _cerrar_bitacora(con, p_id_carga, estado, detalle, leidas, cargadas, rechazadas, corregidas)
    return {"estado": estado, "leidas": leidas, "cargadas": cargadas, "rechazadas": rechazadas, "corregidas": corregidas}

def _cerrar_bitacora(con, id_carga, estado, detalle, leidas, cargadas, rechazadas, corregidas):
    con.execute("UPDATE bitacora_cargas SET fin = ?, estado = ?, detalle = ?, leidas = ?, cargadas = ?, rechazadas = ?, corregidas = ? WHERE id_carga = ?",
                (datetime.now().isoformat(timespec="seconds"), estado, detalle, leidas, cargadas, rechazadas, corregidas, id_carga))

filas_habitual = archivo.to_dict("records")
print("Procedimiento definido ✔")

**Qué vimos:** el procedimiento tiene tres salidas posibles: `OK`, `DETENIDA` (el origen está mal) y `ERROR` (algo inesperado). En las tres, la **bitácora queda escrita** y, si no fue `OK`, **no queda nada cargado a medias**. **Una nota sobre el `COMMIT`:** acá el procedimiento es la operación completa, por eso confirma una sola vez, al final. Si se invocara dentro de una operación mayor, el `COMMIT` iría en quien lo invoca (lo vemos en el Paso 12).

## Paso 6 · Ejecutamos: el archivo habitual

Antes de correr, una predicción: el archivo tiene unos 5,5 % de filas con errores de rechazo. ¿Qué va a hacer el procedimiento?

In [ ]:
resumen = pr_cargar_ordenes(con, 1, "ordenes_20251001.csv", filas_habitual)
pd.Series(resumen, name="resultado de la carga 1").to_frame()

Las **consultas de conciliación** del Paso 2:

In [ ]:
verificar_corrida(con, 1, resumen["leidas"])

**Los rechazos**, con su motivo. ¿Coinciden con los errores plantados?

In [ ]:
rech = pd.read_sql("SELECT motivo, COUNT(*) AS filas FROM rechazos WHERE id_carga = 1 GROUP BY motivo ORDER BY filas DESC", con)
ax = rech.set_index("motivo")["filas"].plot(kind="barh", figsize=(7, 2.8), color="#7C5CF6")
ax.set_title("Filas rechazadas por motivo (carga 1)"); ax.set_xlabel("filas"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
rech

**Qué vimos:** la carga terminó `OK`, los cinco controles pasan y cada fila rechazada tiene su motivo. También se **corrigieron** los canales mal escritos: figuran en `corregidas`, no en los rechazos.

## Paso 7 · Cuando algo sale mal

Dos fallas distintas, dos respuestas distintas. Una predicción antes de cada una.

**Falla 1 · el origen está mal.** Llega un archivo con casi un tercio de filas rechazables.

In [ ]:
antes = con.execute("SELECT COUNT(*) FROM ordenes").fetchone()[0]
predecir = "DETENIDA y sin cambios en la tabla (la corrida se frena y se revierte)"
print("Predicción:", predecir)
r2 = pr_cargar_ordenes(con, 2, "ordenes_20251002.csv", archivo_malo.to_dict("records"))
despues = con.execute("SELECT COUNT(*) FROM ordenes").fetchone()[0]
print("Resultado:", r2["estado"], "· rechazadas:", f"{r2['rechazadas']:,} de {r2['leidas']:,}", f"({r2['rechazadas'] / r2['leidas']:.0%})")
print(f"Órdenes antes: {antes:,} · después: {despues:,} → {'✅ sin cambios' if antes == despues else '❌ cambió'}")

**Falla 2 · algo inesperado a mitad de la corrida** (se corta la conexión, falla un disco). Forzamos una falla en la fila 3.000.

In [ ]:
antes = con.execute("SELECT COUNT(*) FROM ordenes").fetchone()[0]
try:
    pr_cargar_ordenes(con, 3, "ordenes_20251003.csv", filas_habitual, falla_en_fila=3000)
except RuntimeError as e:
    print("La excepción se propagó (no se silenció):", e)
despues = con.execute("SELECT COUNT(*) FROM ordenes").fetchone()[0]
print(f"Órdenes antes: {antes:,} · después: {despues:,} → {'✅ nada quedó a medias' if antes == despues else '❌ quedó cargado a medias'}")

**La bitácora** registra las tres corridas, aunque dos hayan sido revertidas:

In [ ]:
pd.read_sql("SELECT id_carga, archivo, estado, leidas, cargadas, rechazadas, corregidas, detalle FROM bitacora_cargas ORDER BY id_carga", con)

**Qué vimos:** el `ROLLBACK` deshizo toda la carga, pero la **bitácora sigue diciendo qué pasó**. Eso es lo que permite **detectar, contener y recuperar**.

## Paso 8 · Idempotencia: ¿qué pasa si se ejecuta dos veces?

En la vida real, un planificador reintenta, o alguien vuelve a lanzar la carga. Un procedimiento **idempotente** da el mismo resultado ejecutado una vez o diez.

In [ ]:
antes = con.execute("SELECT COUNT(*) FROM ordenes").fetchone()[0]
again = pr_cargar_ordenes(con, 1, "ordenes_20251001.csv", filas_habitual)      # la misma carga de antes
despues = con.execute("SELECT COUNT(*) FROM ordenes").fetchone()[0]
print("Estado:", again["estado"], "· órdenes antes:", f"{antes:,}", "· después:", f"{despues:,}", "→", "✅ no se duplicó nada" if antes == despues else "❌ se duplicó")

**Qué vimos:** el procedimiento mira la bitácora: si esa carga ya terminó `OK`, no hace nada. Y aunque no lo mirara, la **clave primaria** de `ordenes` lo frenaría. Dos defensas distintas para el mismo riesgo.

## Paso 9 · Las pruebas

Una prueba tiene **una entrada, una salida esperada y una salida obtenida**. Cada caso corre sobre una **base nueva**, así no se contaminan entre sí. Probamos el camino feliz **y los bordes**. Para probar las **reglas**, el umbral se deja en 100 %: si no, una fila rechazada de una sola fila ya detendría la corrida (el umbral se prueba aparte, al final).

In [ ]:
def fila(**kw):
    base = {"nro_fila": 1, "id_orden": "OS9000001", "id_cliente": "C0001", "canal": "WEB", "monto": 1000.0, "fecha_alta": "2025-10-05"}
    return {**base, **kw}

def correr(filas, umbral=1.0, **kw):
    """Corre el procedimiento sobre una base nueva. Con umbral=1.0 se prueban las REGLAS sin que el umbral frene la corrida."""
    con_t = nueva_base()
    return con_t, pr_cargar_ordenes(con_t, 99, "prueba.csv", filas, p_umbral=umbral, **kw)

PRUEBAS = [
    ("camino feliz: 3 filas válidas",           [fila(nro_fila=i, id_orden=f"OS90000{i}") for i in (1, 2, 3)], dict(cargadas=3, rechazadas=0)),
    ("canal con espacios: se corrige",          [fila(canal=" web ")],                                         dict(cargadas=1, corregidas=1)),
    ("monto en el límite inferior (1)",         [fila(monto=1)],                                               dict(cargadas=1)),
    ("monto en el límite superior (50.000)",    [fila(monto=50_000)],                                          dict(cargadas=1)),
    ("monto apenas fuera (50.000,01)",          [fila(monto=50_000.01)],                                       dict(rechazadas=1)),
    ("monto apenas fuera (0,99)",               [fila(monto=0.99)],                                            dict(rechazadas=1)),
    ("canal inexistente",                       [fila(canal="FAX")],                                           dict(rechazadas=1)),
    ("cliente inexistente",                     [fila(id_cliente="C9999")],                                    dict(rechazadas=1)),
    ("fecha inválida",                          [fila(fecha_alta="2025-13-40")],                               dict(rechazadas=1)),
    ("orden repetida en el archivo",            [fila(), fila(nro_fila=2)],                                    dict(cargadas=1, rechazadas=1)),
    ("orden ya cargada en la base",             [fila(id_orden="OS0000001")],                                  dict(rechazadas=1)),
    ("archivo vacío",                           [],                                                            dict(cargadas=0, rechazadas=0)),
]
resultados = []
for nombre, filas, esperado in PRUEBAS:
    _, obtenido = correr(filas)
    ok = all(obtenido[k] == v for k, v in esperado.items())
    resultados.append(("✅" if ok else "❌", nombre, esperado, {k: obtenido[k] for k in esperado}))
pd.DataFrame(resultados, columns=["", "caso", "salida esperada", "salida obtenida"])

Y dos pruebas más, sobre **toda la corrida**: el umbral y la falla inesperada.

In [ ]:
# umbral: 2 de 4 filas rechazadas (50 %) supera el 20 % → DETENIDA y sin cargar nada
con_t, r = correr([fila(nro_fila=1, id_orden="OS9000001"), fila(nro_fila=2, id_orden="OS9000002", canal="FAX"),
                   fila(nro_fila=3, id_orden="OS9000003", monto=-5), fila(nro_fila=4, id_orden="OS9000004")], umbral=UMBRAL_RECHAZO)
assert r["estado"] == "DETENIDA" and con_t.execute("SELECT COUNT(*) FROM ordenes WHERE id_carga = 99").fetchone()[0] == 0
print("✅ umbral superado → DETENIDA y no se cargó nada")

# falla inesperada: ERROR, excepción propagada y nada cargado
con_t = nueva_base()
try:
    pr_cargar_ordenes(con_t, 99, "prueba.csv", [fila(nro_fila=i, id_orden=f"OS90000{i}") for i in (1, 2, 3)], falla_en_fila=2)
    assert False, "tenía que propagar la excepción"
except RuntimeError:
    pass
assert con_t.execute("SELECT COUNT(*) FROM ordenes WHERE id_carga = 99").fetchone()[0] == 0
assert con_t.execute("SELECT estado FROM bitacora_cargas WHERE id_carga = 99").fetchone()[0] == "ERROR"
print("✅ falla inesperada → ERROR, excepción propagada, nada cargado y bitácora escrita")

## Paso 10 · Segunda situación: actualizar la prioridad de clientes

Cada noche, NexoServicios recalcula el **score de prioridad** de sus clientes a partir de `stg_clientes` (lo que llegó del CRM) y lo guarda en `dim_cliente`. Operaciones usa ese score para ordenar las órdenes de la mañana. Ya conocés las **tres versiones** del procedimiento `actualizar_prioridad_clientes`: son las de la práctica asincrónica. Acá **no las vamos a explicar: las ponemos a prueba**. Antes de ejecutar cada paso, **predecí qué va a pasar** y anotá tu respuesta; después comprobamos si coincide.

> **Nivel de realización:** el PL/SQL de abajo es **especificado, no ejecutado en este entorno**. El notebook ejecuta equivalentes en Python sobre SQLite, que son los que medimos.

### Versión 1 · El original

```sql
CREATE OR REPLACE PROCEDURE actualizar_prioridad_clientes IS
  CURSOR c_clientes IS
    SELECT id_cliente, antiguedad_meses, tickets_abiertos
      FROM stg_clientes;
BEGIN
  FOR r IN c_clientes LOOP
    UPDATE dim_cliente
       SET score_prioridad    = calcular_score(r.antiguedad_meses, r.tickets_abiertos),
           fecha_actualizacion = SYSDATE
     WHERE id_cliente = r.id_cliente;
    COMMIT;
  END LOOP;
END actualizar_prioridad_clientes;
```

### Versión 2 · Con SQL masivo

```sql
CREATE OR REPLACE PROCEDURE actualizar_prioridad_clientes IS
  v_filas NUMBER;
BEGIN
  MERGE INTO dim_cliente d
  USING (
    SELECT id_cliente,
           CASE
             WHEN tickets_abiertos >= 5 THEN 100
             WHEN antiguedad_meses >= 24 THEN 70
             ELSE 40
           END AS score_calculado
      FROM stg_clientes
  ) s
     ON (d.id_cliente = s.id_cliente)
  WHEN MATCHED THEN
    UPDATE SET d.score_prioridad     = s.score_calculado,
               d.fecha_actualizacion = SYSDATE;

  v_filas := SQL%ROWCOUNT;
  COMMIT;

  INSERT INTO log_procesos (proceso, etapa, filas, fecha)
  VALUES ('actualizar_prioridad_clientes', 'merge_scores', v_filas, SYSDATE);
  COMMIT;
END actualizar_prioridad_clientes;
```

### Versión 3 · Por lotes

```sql
CREATE OR REPLACE PROCEDURE actualizar_prioridad_clientes IS
  TYPE t_ids    IS TABLE OF dim_cliente.id_cliente%TYPE;
  TYPE t_scores IS TABLE OF dim_cliente.score_prioridad%TYPE;

  CURSOR c_clientes IS
    SELECT id_cliente, antiguedad_meses, tickets_abiertos FROM stg_clientes;

  v_ids    t_ids;
  v_scores t_scores := t_scores();
  v_ant    DBMS_SQL.NUMBER_TABLE;
  v_tick   DBMS_SQL.NUMBER_TABLE;
  c_lote   CONSTANT PLS_INTEGER := 5000;
  v_errores PLS_INTEGER;
BEGIN
  OPEN c_clientes;
  LOOP
    FETCH c_clientes BULK COLLECT INTO v_ids, v_ant, v_tick LIMIT c_lote;
    EXIT WHEN v_ids.COUNT = 0;

    v_scores.DELETE;
    v_scores.EXTEND(v_ids.COUNT);
    FOR i IN 1 .. v_ids.COUNT LOOP
      v_scores(i) := calcular_score(v_ant(i), v_tick(i));
    END LOOP;

    BEGIN
      FORALL i IN 1 .. v_ids.COUNT SAVE EXCEPTIONS
        UPDATE dim_cliente
           SET score_prioridad     = v_scores(i),
               fecha_actualizacion = SYSDATE
         WHERE id_cliente = v_ids(i);
    EXCEPTION
      WHEN OTHERS THEN
        v_errores := SQL%BULK_EXCEPTIONS.COUNT;
        FOR j IN 1 .. v_errores LOOP
          INSERT INTO log_errores (proceso, indice_lote, codigo_error, fecha)
          VALUES ('actualizar_prioridad_clientes',
                  SQL%BULK_EXCEPTIONS(j).ERROR_INDEX,
                  SQL%BULK_EXCEPTIONS(j).ERROR_CODE,
                  SYSDATE);
        END LOOP;
    END;
  END LOOP;
  CLOSE c_clientes;
  COMMIT;
END actualizar_prioridad_clientes;
```

### Cómo respondemos las predicciones

Cada predicción tiene **dos celdas**: la primera muestra la pregunta y deja una variable `respuesta = None`. **Cambiala por tu letra** (por ejemplo, `respuesta = "B"`) y corré la segunda, que comprueba tu respuesta. Si no respondés, no revela nada.

In [ ]:
def pregunta(texto, opciones):
    """Muestra una pregunta de predicción con sus opciones."""
    print(texto)
    for letra, opcion in zip("ABCD", opciones):
        print(f"   {letra}) {opcion}")

def comprobar(respuesta, correcta, explicacion):
    """Compara tu respuesta con la correcta y explica por qué."""
    if respuesta is None:
        print('👉 Primero escribí tu respuesta en la celda de arriba (por ejemplo: respuesta = "B") y volvé a correr esta celda.')
        return
    ok = str(respuesta).strip().upper() == correcta
    print(("✅ Correcto. " if ok else f"❌ Todavía no: la respuesta es la {correcta}. ") + explicacion)
print("Ayudas de predicción listas ✔")

### Los equivalentes ejecutables

| Versión | En PL/SQL | Su equivalente acá (Python sobre SQLite) |
| --- | --- | --- |
| **1 · Original** | `UPDATE` por fila y `COMMIT` dentro del bucle | Igual: un `UPDATE` por fila, con un `COMMIT` por fila |
| **2 · SQL masivo** | `MERGE` | Un solo `UPDATE ... FROM` (SQLite no tiene `MERGE`) |
| **3 · Por lotes** | `BULK COLLECT ... LIMIT` + `FORALL` | `fetchmany` + `executemany` por lote |

**Dos supuestos para poder compararlas:** `calcular_score` implementa la **misma regla** que el `CASE` de la versión 2 (5 tickets o más → 100; 24 meses o más de antigüedad → 70; si no, 40), y los lotes son de 500 filas (en el PL/SQL son de 5.000; acá hay solo 2.000 clientes).

In [ ]:
import os
import tempfile

N_CLIENTES = 2_000                    # si tu máquina es lenta, bajalo; si es rápida, subilo
LOTE = 500
DIRECTORIO = tempfile.mkdtemp()       # las bases van a archivos (no a memoria): así cada COMMIT cuesta lo que cuesta de verdad
HOY = lambda: datetime.now().isoformat(timespec="seconds")

def calcular_score(antiguedad_meses, tickets_abiertos):
    """La regla del score (la función calcular_score del procedimiento original)."""
    if tickets_abiertos >= 5:
        return 100
    if antiguedad_meses >= 24:
        return 70
    return 40

class ConexionMedida:
    """Envuelve la conexión y cuenta cuántas veces se le habla a la base y cuántos COMMIT se hacen: la evidencia que NO depende de tu máquina."""
    def __init__(self, con):
        self.con, self.llamadas, self.commits = con, 0, 0
    def execute(self, sql, params=()):
        self.llamadas += 1
        if sql.strip().upper().startswith("COMMIT"):
            self.commits += 1
        return self.con.execute(sql, params)
    def executemany(self, sql, filas):
        self.llamadas += 1
        return self.con.executemany(sql, filas)

def nueva_base_prioridad(n=N_CLIENTES, funcion_score=calcular_score):
    """stg_clientes (lo que llegó del CRM) y dim_cliente (donde se guarda el score). 1 % de los clientes de stg no existe en dim_cliente."""
    g = np.random.default_rng(5)
    ids = [f"C{i:06d}" for i in range(1, n + 1)]
    stg = pd.DataFrame({"id_cliente": ids, "antiguedad_meses": g.integers(1, 120, size=n), "tickets_abiertos": g.poisson(1.5, size=n)})
    sin_pareja = pd.DataFrame({"id_cliente": [f"X{i:05d}" for i in range(max(1, n // 100))], "antiguedad_meses": 10, "tickets_abiertos": 0})
    con_ = sqlite3.connect(os.path.join(DIRECTORIO, f"base_{time.time_ns()}.db"), isolation_level=None)
    con_.create_function("calcular_score", 2, funcion_score)       # la función queda disponible dentro del SQL, como en Oracle
    con_.executescript("""
        CREATE TABLE stg_clientes (id_cliente TEXT, antiguedad_meses INTEGER, tickets_abiertos INTEGER);
        CREATE TABLE dim_cliente (id_cliente TEXT PRIMARY KEY,
                                  score_prioridad INTEGER CHECK (score_prioridad BETWEEN 0 AND 100), fecha_actualizacion TEXT);
        CREATE TABLE log_procesos (proceso TEXT, etapa TEXT, filas INTEGER, fecha TEXT);""")
    pd.concat([stg, sin_pareja], ignore_index=True).to_sql("stg_clientes", con_, if_exists="append", index=False)
    pd.DataFrame({"id_cliente": ids, "score_prioridad": 0, "fecha_actualizacion": None}).to_sql("dim_cliente", con_, if_exists="append", index=False)
    return ConexionMedida(con_)

def score_final(cm):
    return pd.read_sql("SELECT id_cliente, score_prioridad FROM dim_cliente ORDER BY id_cliente", cm.con)
print(f"Listo: {N_CLIENTES:,} clientes en dim_cliente y {N_CLIENTES + max(1, N_CLIENTES // 100):,} filas en stg_clientes.")

In [ ]:
# --- Versión 1 · el original: un UPDATE por fila y un COMMIT por fila ---
def v1_original(c, falla_en_fila=None):
    filas = c.execute("SELECT id_cliente, antiguedad_meses, tickets_abiertos FROM stg_clientes").fetchall()
    for i, (id_cliente, ant, tick) in enumerate(filas, 1):          # FOR r IN c_clientes LOOP
        if falla_en_fila == i:
            raise RuntimeError(f"falla inesperada en la fila {i}")
        c.execute("BEGIN")
        c.execute("UPDATE dim_cliente SET score_prioridad = calcular_score(?, ?), fecha_actualizacion = ? WHERE id_cliente = ?", (ant, tick, HOY(), id_cliente))
        c.execute("COMMIT")                                         # ❌ un COMMIT por fila

# --- Lo mismo, cambiando UNA sola cosa: el COMMIT pasa al final ---
def v1b_un_commit(c, falla_en_fila=None):
    filas = c.execute("SELECT id_cliente, antiguedad_meses, tickets_abiertos FROM stg_clientes").fetchall()
    c.execute("BEGIN")
    try:
        for i, (id_cliente, ant, tick) in enumerate(filas, 1):
            if falla_en_fila == i:
                raise RuntimeError(f"falla inesperada en la fila {i}")
            c.execute("UPDATE dim_cliente SET score_prioridad = calcular_score(?, ?), fecha_actualizacion = ? WHERE id_cliente = ?", (ant, tick, HOY(), id_cliente))
    except Exception:
        c.execute("ROLLBACK")                                       # si algo falla, vuelve todo atrás
        raise
    c.execute("COMMIT")                                             # ✅ uno solo, al final

# --- Versión 2 · SQL masivo: una sola sentencia (equivalente de MERGE) ---
def v2_masivo(c):
    c.execute("BEGIN")
    c.execute("""UPDATE dim_cliente SET score_prioridad = s.score_calculado, fecha_actualizacion = ?
                   FROM (SELECT id_cliente,
                                CASE WHEN tickets_abiertos >= 5 THEN 100 WHEN antiguedad_meses >= 24 THEN 70 ELSE 40 END AS score_calculado
                           FROM stg_clientes) AS s
                  WHERE dim_cliente.id_cliente = s.id_cliente""", (HOY(),))      # la regla se REESCRIBE en SQL (el CASE)
    c.execute("INSERT INTO log_procesos VALUES ('actualizar_prioridad_clientes', 'merge_scores', (SELECT changes()), ?)", (HOY(),))
    c.execute("COMMIT")

# --- Versión 3 · por lotes (equivalente de BULK COLLECT ... LIMIT + FORALL) ---
def v3_lotes(c, lote=LOTE):
    cursor = c.con.execute("SELECT id_cliente, antiguedad_meses, tickets_abiertos FROM stg_clientes")
    c.llamadas += 1
    c.execute("BEGIN")
    while True:
        bloque = cursor.fetchmany(lote); c.llamadas += 1           # BULK COLLECT ... LIMIT
        if not bloque:
            break
        datos = [(calcular_score(ant, tick), HOY(), idc) for idc, ant, tick in bloque]
        c.executemany("UPDATE dim_cliente SET score_prioridad = ?, fecha_actualizacion = ? WHERE id_cliente = ?", datos)   # FORALL
    c.execute("COMMIT")                                             # uno solo, al final
print("Las tres versiones (y la variante con un solo COMMIT) están definidas ✔")

## Paso 11 · El original: ¿qué está mal?

Antes de medir nada, **leé la versión 1** y respondé: ¿qué pasa si el proceso se cae a mitad de la corrida? ¿Qué le avisa a quien lo opera? Lo comprobamos con una falla simulada en la fila 1.200:

In [ ]:
pregunta("PREDICCIÓN 1 · En la versión 1 (el original), el proceso se cae en la fila 1.200. ¿Cuántos clientes quedan con el score nuevo?",
         ["0: nada queda a medias", "1.199: las filas anteriores ya estaban confirmadas", "Todos: el cursor ya había leído todo", "Depende de cuándo se confirme el lote"])
respuesta = None        # 👉 TU TURNO: cambiá None por tu letra, por ejemplo "B"

In [ ]:
comprobar(respuesta, "B", "Cada COMMIT confirma su fila para siempre: las 1.199 anteriores ya no se pueden deshacer. Es lo que vamos a verificar ahora.")

In [ ]:
for nombre, version in (("1 · original (un COMMIT por fila)", v1_original), ("1b · mismo código, un solo COMMIT al final", v1b_un_commit)):
    cm = nueva_base_prioridad()
    try:
        version(cm, falla_en_fila=1200)
    except RuntimeError as e:
        print(f"{nombre}\n   se cayó: {e}")
    actualizadas = cm.con.execute("SELECT COUNT(*) FROM dim_cliente WHERE score_prioridad <> 0").fetchone()[0]
    print(f"   clientes que quedaron con el score nuevo: {actualizadas:,}  →  {'⚠️ actualizados a medias, ya confirmados' if actualizadas else '✅ nada a medias'}")

**Una cosa más.** En `stg_clientes` llegaron algunos clientes que **no existen** en `dim_cliente`. Antes de comprobarlo, otra predicción:

In [ ]:
pregunta("PREDICCIÓN 2 · Una fila de stg_clientes no tiene pareja en dim_cliente. ¿Qué hacen las versiones 1 y 2 con ella?",
         ["Fallan con un error", "La insertan en dim_cliente", "No actualizan nada y no avisan"])
respuesta = None        # 👉 TU TURNO

In [ ]:
comprobar(respuesta, "C", "El UPDATE (y el MERGE con WHEN MATCHED) simplemente no encuentra la fila: no falla y nadie se entera. Lo contamos abajo.")

In [ ]:
cm = nueva_base_prioridad()
sin_pareja = cm.con.execute("SELECT COUNT(*) FROM stg_clientes s WHERE NOT EXISTS (SELECT 1 FROM dim_cliente d WHERE d.id_cliente = s.id_cliente)").fetchone()[0]
total = cm.con.execute("SELECT COUNT(*) FROM stg_clientes").fetchone()[0]
print(f"{sin_pareja:,} de {total:,} filas de stg_clientes no tienen pareja en dim_cliente y se pierden en silencio.")

**Qué vimos:** el `COMMIT` dentro del bucle deja la tabla **actualizada a medias** si algo falla, y ninguna versión **concilia** lo que leyó contra lo que actualizó.

## Paso 12 · Medir antes de optimizar

Optimizar sin método produce cambios que no se pueden evaluar ni revertir con certeza. El ciclo es: **medir**, **localizar** dónde está el costo, formular una **hipótesis** y **cambiar una sola cosa**. Después, volver a medir.

**Síntoma:** el original es lento. **Evidencia a mirar:** no solo el tiempo, también *cuántas veces* le habla a la base y *cuántos COMMIT* hace. Medimos las cuatro variantes sobre la misma base de partida:

In [ ]:
pregunta("PREDICCIÓN 3 · Tomamos la versión 1 y cambiamos UNA sola cosa: el COMMIT pasa al final, fuera del bucle. ¿Qué esperás?",
         ["El tiempo no cambia", "Baja un poco", "Baja muchísimo y además la corrida pasa a ser atómica", "Queda igual de lenta, pero atómica"])
respuesta = None        # 👉 TU TURNO

In [ ]:
comprobar(respuesta, "C", "Cada COMMIT tiene un costo, y con uno solo la corrida se confirma o se revierte entera. Lo medimos ahora.")

In [ ]:
variantes = (("1 · original (COMMIT por fila)", v1_original), ("1b · un solo COMMIT (cambio: solo el COMMIT)", v1b_un_commit),
             ("2 · SQL masivo", v2_masivo), ("3 · por lotes", v3_lotes))
filas, finales = [], {}
for nombre, version in variantes:
    cm = nueva_base_prioridad()
    t0 = time.perf_counter()
    version(cm)
    seg = time.perf_counter() - t0
    finales[nombre] = score_final(cm)
    filas.append((nombre, seg, cm.llamadas, cm.commits))
medicion = pd.DataFrame(filas, columns=["variante", "segundos", "llamadas a la base", "COMMIT"]).set_index("variante")
ref = finales[variantes[0][0]]
medicion["mismo resultado que 1"] = ["✅" if ref.equals(finales[n]) else "❌" for n in medicion.index]
medicion["veces más rápida que 1"] = (medicion.loc[variantes[0][0], "segundos"] / medicion["segundos"]).round(1)
medicion

In [ ]:
ax = medicion["segundos"].plot(kind="barh", figsize=(7.5, 2.8), color=["#F59E0B", "#7C5CF6", "#10D9A0", "#10D9A0"], logx=True)
ax.set_title("Tiempo de cada variante (escala logarítmica: menos es mejor)"); ax.set_xlabel("segundos")
ax.invert_yaxis(); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

**Cómo leer esto:**
- **Localizar:** la variante 1 hace miles de COMMIT; la 1b hace uno. Es **la única diferencia** entre las dos, y el tiempo cae enormemente. La causa de la lentitud no estaba en el cálculo: estaba en **dónde confirmás**.
- **Una variable a la vez:** si hubiéramos pasado directo a la versión 2 o 3, no sabríamos cuánto del beneficio viene del `COMMIT` y cuánto del SQL masivo.
- **Los números dependen de tu máquina** y de este tamaño de datos. Lo que se sostiene es el orden y la proporción, y los conteos de llamadas y COMMIT, que **no** dependen de la máquina.
- **Y el cambio también arregla la atomicidad:** con un solo `COMMIT`, la falla de la fila 1.200 dejaba cero filas a medias.

In [ ]:
pregunta("PREDICCIÓN 4 · ¿Cuál de las tres versiones tiene la regla del score escrita en un lugar distinto de las otras dos?",
         ["La versión 1", "La versión 2", "La versión 3"])
respuesta = None        # 👉 TU TURNO

In [ ]:
comprobar(respuesta, "B", "La versión 2 reescribe la regla en un CASE; las versiones 1 y 3 usan calcular_score. Veamos qué cuesta eso.")

### Una regla en dos lugares

La versión 2 **reescribe la regla** del score en un `CASE`; las versiones 1 y 3 la toman de `calcular_score`. Mientras las dos coincidan, nadie lo nota. ¿Qué pasa el día que alguien cambia **una sola**? Supongamos que la regla nueva pide **12 meses** de antigüedad, en lugar de 24, y se actualiza la función pero nadie toca el `CASE`:

In [ ]:
def calcular_score_nueva(antiguedad_meses, tickets_abiertos):
    if tickets_abiertos >= 5:
        return 100
    if antiguedad_meses >= 12:                 # la regla cambió: 12 meses en lugar de 24
        return 70
    return 40

cm_a = nueva_base_prioridad(funcion_score=calcular_score_nueva); v1b_un_commit(cm_a)    # usa la función (regla nueva)
cm_b = nueva_base_prioridad(funcion_score=calcular_score_nueva); v2_masivo(cm_b)        # usa su propio CASE (regla vieja)
distintos = (score_final(cm_a)["score_prioridad"] != score_final(cm_b)["score_prioridad"]).sum()
print(f"Clientes con un score distinto según qué versión se ejecute: {distintos:,} de {N_CLIENTES:,} ({distintos / N_CLIENTES:.0%})")

**Qué vimos:** la versión más rápida tiene un riesgo de **gobierno**: la regla de negocio vive en dos lugares y puede divergir sin que nada falle. Una decisión de optimización tiene un **costo** que hay que declarar.

### Una nota sobre el `COMMIT`: ¿quién confirma?

En este notebook, cada procedimiento es **la operación completa** (una carga, una actualización nocturna), así que confirma **una sola vez, al final**. Pero si un procedimiento se invoca **dentro de una operación mayor**, un `COMMIT` interno confirma los cambios sin que quien lo invoca pueda impedirlo: rompe la atomicidad de la operación grande.

| Camino | Dónde está el `COMMIT` | Qué pasa |
| --- | --- | --- |
| **A** | Dentro del procedimiento | El invocador pierde el control: no puede revertir lo que ya se confirmó |
| **B** | En quien lo invoca | El invocador combina varios procedimientos en una sola unidad atómica y revierte todo si algo falla |

**La regla:** el `COMMIT` debe vivir en el nivel más alto que tenga visibilidad completa de la operación.

## Ejercicio por equipos · tres versiones, una decisión

Cada equipo (de 3 o 4 personas, con los roles de **arquitectura, operación, gobierno y revisión**) trabaja sobre las **tres versiones** de `actualizar_prioridad_clientes`, que ya verificaron. NexoServicios las corre **cada noche, con 200.000 clientes**. Hay que decidir cuál usar y dejarla lista para operar.

**Parte 1 · Comparar.** Completá la tabla con las tres versiones. Ninguna tiene todavía un contrato escrito: eso también cuenta.

**Parte 2 · Decidir.** Elegí una versión para el proceso nocturno y completá el **registro de decisión**, con una **alternativa descartada** y **qué medirías** para confirmar la decisión (el ciclo: medir, localizar, hipótesis, cambiar).

**Parte 3 · Asegurar.** Escribí el **contrato de la versión elegida** en cinco partes y **una prueba de borde**.

> Si te preguntás «¿está bien?», la pregunta es otra: **¿qué restricción satisface y cómo comprobarías que la satisface?**

**La entrega responde seis preguntas:** ¿qué problema y decisión resuelve? ¿qué entra y qué queda fuera de alcance? ¿qué alternativa se descartó y por qué? ¿cómo se prueba antes de pasar a producción? ¿cómo se detecta, contiene y recupera una falla? ¿quién decide cuando un control no pasa?

### Parte 1 · La tabla a completar

In [ ]:
preguntas = ["¿Cuál es la unidad de trabajo? (una fila, un lote, todo junto)", "¿Dónde está el COMMIT y qué pasa si falla a mitad?",
             "¿Cómo maneja los errores?", "¿Qué queda registrado de la corrida?", "¿Dónde vive la regla del score?",
             "¿Se puede volver a ejecutar sin daño?"]
comparacion = pd.DataFrame("", index=preguntas, columns=["1 · Original", "2 · SQL masivo", "3 · Por lotes"])
comparacion          # 👉 TU TURNO: completá cada celda, por ejemplo: comparacion.loc[preguntas[0], "1 · Original"] = "una fila"

### Parte 2 · Para probar tus hipótesis
`correr_version` ejecuta una variante sobre una base nueva y devuelve lo que necesitás para decidir: **tiempo, llamadas, COMMIT y la tabla final**. Cambiá una sola cosa por vez.

In [ ]:
def correr_version(version, **kw):
    """Corre una versión sobre una base nueva. Devuelve (tiempo en segundos, llamadas a la base, COMMIT, tabla final)."""
    cm = nueva_base_prioridad()
    t0 = time.perf_counter()
    version(cm, **kw)
    return round(time.perf_counter() - t0, 4), cm.llamadas, cm.commits, score_final(cm)

# 👉 TU TURNO · por ejemplo: ¿cambia algo si el lote es de 100 filas en lugar de 500?
# seg, llamadas, commits, tabla = correr_version(v3_lotes, lote=100)
# print(seg, llamadas, commits)

**Registro de decisión** (la plantilla del programa). Completalo para tu versión elegida:

| Campo | Tu respuesta |
| --- | --- |
| **Problema que se resuelve** | |
| **Decisión que habilita** | |
| **Restricciones verificadas** | |
| **Supuestos por confirmar** | |
| **Opción elegida** y su **beneficio** | |
| **Alternativa descartada** y su **costo o riesgo** | |
| **Trade-off aceptado** | |
| **Control** (cómo se detecta que algo falla) | |
| **Señal para revisar esta decisión** | |
| **Responsable de revisión** | |

### Parte 3 · El contrato de la versión elegida

| Componente | Qué se escribe |
| --- | --- |
| **1 · Parámetros** | Qué recibe (`IN`), qué devuelve (`OUT`) y qué hace ambas cosas (`IN OUT`). Tipos explícitos |
| **2 · Precondiciones** | Qué debe ser verdadero antes de invocarlo |
| **3 · Resultado** | Qué estado queda en la base cuando termina sin error: efecto observable y verificable |
| **4 · Errores esperados** | Qué excepciones pueden ocurrir, cuáles se manejan adentro y cuáles se propagan |
| **5 · Efectos** | Qué tablas se modifican, qué registros de auditoría se generan |

In [ ]:
# 👉 TU TURNO · una prueba de borde de tu versión elegida: entrada, salida esperada y salida obtenida